In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import time

CATALOG = "fahrlytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

spark.sql(f"USE CATALOG {CATALOG}")

VALID_CITIES = ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne"]

print(f"Catalog : {CATALOG}")
print(f"Source  : {BRONZE}.clickstream")
print(f"Target  : {SILVER}.clickstream")

In [0]:
print("Evaluating load strategy...")

table_exists = spark.catalog.tableExists(f"{SILVER}.clickstream")
is_empty = not table_exists or spark.table(f"{SILVER}.clickstream").count() == 0

if is_empty:
    LOAD_TYPE = "full"
    df_bronze = spark.table(f"{BRONZE}.clickstream")
    print("Initiating FULL load for clickstream events.")
else:
    LOAD_TYPE = "incremental"
    last_run = spark.sql(f"SELECT MAX(_processed_at) FROM {SILVER}.clickstream").collect()[0][0]
    df_bronze = spark.table(f"{BRONZE}.clickstream").filter(F.col("_ingested_at") > last_run)
    print(f"Initiating INCREMENTAL load for events ingested after: {last_run}")

input_count = df_bronze.count()
print(f"Incoming event rows: {input_count:,}")

if input_count == 0:
    print("No new clickstream rows to process. Exiting.")
    dbutils.notebook.exit("No new clickstream events")

In [0]:
print("Deduplicating and standardizing event formats...")

df_deduped = df_bronze.dropDuplicates(["event_id"])
dedup_removed = input_count - df_deduped.count()
print(f"Deduplicated count: {df_deduped.count():,} (Dropped {dedup_removed:,} duplicate keys)")

df_base = (
    df_deduped
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
    .withColumn("device_type", F.trim(F.col("device_type")))
    .withColumn("ab_test_group", F.lower(F.trim(F.col("ab_test_group"))))
    .withColumn("ab_test_experiment", F.coalesce(F.col("ab_test_experiment"), F.lit("dynamic_pricing_de_v1")))
)

In [0]:
print("Computing windowed session ordering and step latencies...")

# Window over session sorted chronologically
session_order_window = Window.partitionBy("session_id").orderBy(F.col("event_timestamp").asc(), F.col("event_id").asc())
session_full_window  = Window.partitionBy("session_id")

df_session_metrics = (
    df_base
    # 1. Step sequence number within session (1, 2, 3...)
    .withColumn("event_seq_num", F.row_number().over(session_order_window))
    
    # 2. Total events in session
    .withColumn("total_session_events", F.count("event_id").over(session_full_window))
    
    # 3. Lag timestamp to compute elapsed seconds from previous action
    .withColumn("prev_event_timestamp", F.lag("event_timestamp", 1).over(session_order_window))
    .withColumn("time_since_prev_event_sec", 
        F.when(F.col("prev_event_timestamp").isNotNull(),
               F.unix_timestamp("event_timestamp") - F.unix_timestamp("prev_event_timestamp"))
        .otherwise(0)
    )
    
    # 4. Total session duration (max ts - min ts)
    .withColumn("session_start_time", F.min("event_timestamp").over(session_full_window))
    .withColumn("session_end_time", F.max("event_timestamp").over(session_full_window))
    .withColumn("session_duration_sec", 
        F.unix_timestamp("session_end_time") - F.unix_timestamp("session_start_time")
    )
    .drop("prev_event_timestamp", "session_start_time", "session_end_time")
)

In [0]:
print("Mapping funnel progression flags...")

df_funnel = (
    df_session_metrics
    # Funnel milestone flags
    .withColumn("is_app_open", F.col("event_type") == "app_open")
    .withColumn("is_search", F.col("event_type") == "search_ride")
    .withColumn("is_quote_view", F.col("event_type").isin("view_fare_quote", "view_price"))
    .withColumn("is_booking_confirmed", F.col("event_type") == "confirm_booking")
    .withColumn("is_booking_cancelled", F.col("event_type") == "cancel_booking")
    
    # Bounce check: single-event session that opened and abandoned immediately
    .withColumn("is_bounce_session", (F.col("total_session_events") == 1) & (F.col("is_app_open") == True))
    
    # Telemetry health audit: Flag negative step deltas (clock skew)
    .withColumn("is_valid_step_latency", F.col("time_since_prev_event_sec") >= 0)
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.clickstream (
        event_id                   STRING,
        customer_id                STRING,
        session_id                 STRING,
        city                       STRING,
        device_type                STRING,
        event_type                 STRING,
        event_timestamp            TIMESTAMP,
        event_date                 DATE,
        event_seq_num              INT,
        total_session_events       INT,
        time_since_prev_event_sec  INT,
        session_duration_sec       INT,
        ab_test_group              STRING,
        ab_test_experiment         STRING,
        is_app_open                BOOLEAN,
        is_search                  BOOLEAN,
        is_quote_view              BOOLEAN,
        is_booking_confirmed       BOOLEAN,
        is_booking_cancelled       BOOLEAN,
        is_bounce_session          BOOLEAN,
        is_valid_step_latency      BOOLEAN,
        _ingested_at               TIMESTAMP,
        _processed_at              TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (event_type)
    COMMENT 'Silver clickstream telemetry with session ordering and funnel markers'
""")

In [0]:
print(f"Writing to {SILVER}.clickstream (Mode: {LOAD_TYPE})...")
t0 = time.time()

if LOAD_TYPE == "full":
    (
        df_funnel.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_type")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.clickstream")
    )
    print(f"Full Silver clickstream load written: {df_funnel.count():,} rows")
else:
    target_clickstream = DeltaTable.forName(spark, f"{SILVER}.clickstream")
    (
        target_clickstream.alias("target")
        .merge(
            df_funnel.alias("source"),
            "target.event_type = source.event_type AND target.event_id = source.event_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Incremental clickstream merge complete.")

print(f"Write duration: {round(time.time() - t0, 2)}s")

# File compaction
print("Running Delta OPTIMIZE...")
spark.sql(f"OPTIMIZE {SILVER}.clickstream")
print("Delta OPTIMIZE completed.")

In [0]:
print("=== FAHRLYTICS SILVER CLICKSTREAM AUDIT ===\n")

df_verify = spark.table(f"{SILVER}.clickstream")
total_events = df_verify.count()
print(f"Total events in Silver: {total_events:,}\n")

print("Event Type Distribution:")
df_verify.groupBy("event_type").count().orderBy("count", ascending=False).show()

print("A/B Experiment Group Split:")
df_verify.groupBy("ab_test_group").count().show()

print("Funnel Drop-off Summary:")
df_verify.selectExpr(
    "count(distinct case when is_app_open then session_id end) as sessions_opened",
    "count(distinct case when is_search then session_id end) as sessions_searched",
    "count(distinct case when is_quote_view then session_id end) as sessions_viewed_quote",
    "count(distinct case when is_booking_confirmed then session_id end) as sessions_confirmed",
    "count(distinct case when is_booking_cancelled then session_id end) as sessions_cancelled",
    "count(distinct case when is_bounce_session then session_id end) as sessions_bounced"
).show()

print("Session Duration Averages (Seconds):")
df_verify.select(
    F.round(F.avg("session_duration_sec"), 1).alias("avg_session_duration_sec"),
    F.round(F.avg("total_session_events"), 1).alias("avg_events_per_session")
).show()